In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_5.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

# images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"]
# ...

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    
]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions1 = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries = []
camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    points_cam = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_cam)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries.append(pcd1)
    geometries.append(camera_frame)


    # PLY += pcd1

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


PointCloud with 1015280 points.

In [14]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/maritime1.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/maritime/frame_000406.jpg",
    "/home/tong/recordings/maritime/frame_000454.jpg",
    "/home/tong/recordings/maritime/frame_000501.jpg",
    "/home/tong/recordings/maritime/frame_000541.jpg",
    "/home/tong/recordings/maritime/frame_000564.jpg",
]

views = load_images(images)

# Run inference
predictions1 = model.infer(
    views,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

geometries = []
camera_positions = []

# --- SET YOUR FILTER DISTANCE HERE ---
# This will remove points farther than this distance (in meters/units)
# from the camera's origin.
MAX_FILTER_DISTANCE = 10.0 
print(f"Filtering points farther than {MAX_FILTER_DISTANCE} units from their camera.")
# ----------------------------------------

PLY = o3d.geometry.PointCloud() # Use this for saving a combined cloud

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    # Get raw points, colors, and the pose
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    # --- START: New Filtering Logic ---
    
    # 1. Get the camera's origin (its position in the world frame)
    camera_origin = camera_pose[:3, 3]
    
    # 2. Calculate the distance of each point from the camera origin
    distances = np.linalg.norm(points_world - camera_origin, axis=1)
    
    # 3. Create a boolean mask for points *within* the distance
    mask = distances <= MAX_FILTER_DISTANCE
    
    # 4. Apply the mask to the points and colors
    filtered_points = points_world[mask]
    filtered_colors = colors[mask]
    
    # --- END: New Filtering Logic ---

    # 5. Create the point cloud from the *filtered* data
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(filtered_points)
    pcd1.colors = o3d.utility.Vector3dVector(filtered_colors)
    
    # Get camera center for drawing the path
    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    # Create the camera coordinate frame visualization
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    # Add the filtered cloud and camera frame
    geometries.append(pcd1)
    geometries.append(camera_frame)

    PLY += pcd1 # Add to the combined cloud for saving

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()

# Draw the camera path
if len(camera_positions) > 1:
    line_points = o3d.utility.Vector3dVector(camera_positions)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    camera_path.paint_uniform_color([1, 0, 0]) # Red path
    
    geometries.append(camera_path)

# Apply the final coordinate transform
transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)

# 4. Display all the geometries together in one window
print(f"Displaying combined scene with {len(predictions1)} filtered point clouds...")
o3d.visualization.draw_geometries(geometries)


print(f"Saving to {output_filename}...")
PLY.transform(transform_matrix)
o3d.io.write_point_cloud(output_filename, PLY)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Filtering points farther than 10.0 units from their camera.
Displaying combined scene with 5 filtered point clouds...
Saving to /home/tong/recordings/PLYs/maritime1.ply...


True

In [4]:
o3d.visualization.draw_geometries(geometries)

In [16]:
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    

]

views2 = load_images(batch2)

# Run inference (this will process all images in the list)
preds2 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

last_pose2 = None

PLY = o3d.geometry.PointCloud()


for i, pred in enumerate(preds2):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose2 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [14]:
o3d.visualization.draw_geometries(geometries)

In [17]:
batch3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
    

]

views3 = load_images(batch3)

# Run inference (this will process all images in the list)
preds3 = model.infer(
    views3,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()
last_pose3 = None

for i, pred in enumerate(preds3):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose2@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose3 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)


# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [18]:
o3d.visualization.draw_geometries(geometries)

In [20]:
batch1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",

]


views1 = load_images(batch1)

# Run inference (this will process all images in the list)
preds1 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

geometries = []
last_pose = None
pcd_batch1_overlap = None # We need to save this cloud

print("Processing Batch 1...")
for i, pred1 in enumerate(preds1):
    
    # Use 'pts3d' - these are already in batch 1's world frame
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_world)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    # Add camera pose for visualization
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    geometries.append(pcd1)
    geometries.append(camera_frame)

    if i == len(preds1) - 1:
        # This is frame_000200.jpg in World 1
        pcd_batch1_overlap = pcd1 
        last_pose = camera_pose # Save for visualization if needed

# --- BATCH 2 ---
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",

]

views2 = load_images(batch2)
preds2 = model.infer(views2)
# ...

pcd_batch2_overlap = None # This will be frame_000200.jpg in World 2
point_clouds_batch2 = []  # Store batch 2 clouds temporarily
camera_frames_batch2 = [] # Store batch 2 frames temporarily

print("Processing Batch 2 (in its own coordinate system)...")
for i, pred in enumerate(preds2):
    
    # Use 'pts3d' - these are in batch 2's world frame
    points_world_b2 = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors_b2 = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_world_b2)
    pcd.colors = o3d.utility.Vector3dVector(colors_b2)
    
    camera_pose_b2 = pred["camera_poses"].squeeze().cpu().numpy()
    camera_frame_b2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame_b2.transform(camera_pose_b2)

    if i == 0:
        # This is frame_000200.jpg in World 2
        pcd_batch2_overlap = pcd 
    else:
        # These are the *new* clouds (250, 300, 350, 400)
        point_clouds_batch2.append(pcd)
        camera_frames_batch2.append(camera_frame_b2)

# --- ALIGNMENT STEP (using ICP) ---

print("Aligning Batch 2 to Batch 1 using ICP...")
# Voxel downsample for faster ICP
voxel_size = 0.05 # Adjust this based on your scene's scale
source = pcd_batch2_overlap.voxel_down_sample(voxel_size)
target = pcd_batch1_overlap.voxel_down_sample(voxel_size)

# Set an initial guess (identity matrix)
trans_init = np.identity(4)

# Run ICP
# You may need to tune 'max_correspondence_distance'
reg_p2p = o3d.pipelines.registration.registration_icp(
    source, target, 0.2, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))

# Get the transformation matrix T that maps World 2 -> World 1
T_batch2_to_batch1 = reg_p2p.transformation
print("ICP transformation found:")
print(T_batch2_to_batch1)


# --- ADD BATCH 2 GEOMETRIES (NOW TRANSFORMED) ---

print("Applying transformation to Batch 2...")
for pcd in point_clouds_batch2:
    pcd.transform(T_batch2_to_batch1)
    pcd.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(pcd)

for frame in camera_frames_batch2:
    frame.transform(T_batch2_to_batch1)
    frame.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(frame)

# Apply the Y/Z flip to batch 1 geometries
for i in range(len(predictions1) * 2): # 2 geometries (pcd, frame) per image
    geometries[i].transform(transform_matrix)

# --- VISUALIZE ---
print("Visualizing combined map...")
o3d.visualization.draw_geometries(geometries)

# # --- SAVE ---
# print(f"Saving combined PLY to {output_filename}...")
# combined_pcd = o3d.geometry.PointCloud()
# for geo in geometries:
#     if isinstance(geo, o3d.geometry.PointCloud):
#         combined_pcd += geo

# # Voxel downsample the final cloud for a reasonable file size
# final_pcd = combined_pcd.voxel_down_sample(voxel_size=0.02)
# o3d.io.write_point_cloud(output_filename, final_pcd)
# print("Done.")

Processing Batch 1...
Processing Batch 2 (in its own coordinate system)...
Aligning Batch 2 to Batch 1 using ICP...
ICP transformation found:
[[ 0.99999367  0.00335097 -0.00119958  0.07746477]
 [-0.00335421  0.9999907  -0.00271146  0.05520372]
 [ 0.00119049  0.00271546  0.9999956   0.06450189]
 [ 0.          0.          0.          1.        ]]
Applying transformation to Batch 2...
Visualizing combined map...


In [2]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# Get inference device
# output_filename = "/home/tong/recordings/PLYs/maritime1.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [980.21, 0.0, 825.18],
    [0.0, 980.21, 627.85],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [7]:
# 1. Provide a list of multiple image paths
batch1 = [
    "/home/tong/recordings/moreRocks2_1/frame_0000.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0006.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0012.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0018.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0024.jpg",
]



views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
# Run inference
predictions1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

geometries = []
camera_positions = []
poses = []

# --- SET YOUR FILTER DISTANCE HERE ---
# This will remove points farther than this distance (in meters/units)
# from the camera's origin.
MAX_FILTER_DISTANCE = 15.0 


for i, pred1 in enumerate(predictions1):
    
    # Get raw points, colors, and the pose
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    # --- Filtering---
    camera_origin = camera_pose[:3, 3]
    distances = np.linalg.norm(points_world - camera_origin, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_world[mask]
    filtered_colors = colors[mask]
    

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(filtered_points)
    pcd1.colors = o3d.utility.Vector3dVector(filtered_colors)
    
   
    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    

    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    geometries.append(pcd1)
    geometries.append(camera_frame)
    poses.append(camera_pose)

   

# Draw the camera path
if len(camera_positions) > 1:
    line_points = o3d.utility.Vector3dVector(camera_positions)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    camera_path.paint_uniform_color([1, 0, 0]) # Red path
    
    geometries.append(camera_path)

# Apply the final coordinate transform


for geometry in geometries:
    geometry.transform(transform_matrix)


In [8]:
o3d.visualization.draw_geometries(geometries)

In [10]:
poses

[array([[ 1.0000000e+00, -5.0653623e-05,  7.5918681e-05, -4.8263806e-05],
        [ 5.0648538e-05,  1.0000000e+00,  6.6988803e-05,  6.2632456e-04],
        [-7.5922071e-05, -6.6984961e-05,  1.0000000e+00,  5.7383287e-03],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00]],
       dtype=float32),
 array([[ 0.999934  , -0.00219203, -0.01127793, -0.16167039],
        [ 0.00231355,  0.99993926,  0.01077397, -0.01310315],
        [ 0.01125363, -0.01079935,  0.99987835, -0.02026724],
        [ 0.        ,  0.        ,  0.        ,  1.        ]],
       dtype=float32),
 array([[ 9.9951804e-01, -4.3585775e-03, -3.0735286e-02, -2.0086627e-01],
        [ 5.0907293e-03,  9.9970418e-01,  2.3783347e-02,  6.9626124e-04],
        [ 3.0622533e-02, -2.3928350e-02,  9.9924457e-01, -3.7390497e-02],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.0000000e+00]],
       dtype=float32),
 array([[ 0.99888474, -0.01604552, -0.04440454, -0.30617565],
        [ 0.01803264,  0.9

In [13]:
poses[-4:][0]

array([[ 0.999934  , -0.00219203, -0.01127793, -0.16167039],
       [ 0.00231355,  0.99993926,  0.01077397, -0.01310315],
       [ 0.01125363, -0.01079935,  0.99987835, -0.02026724],
       [ 0.        ,  0.        ,  0.        ,  1.        ]],
      dtype=float32)

In [ ]:
batch2 = [
    "/home/tong/recordings/moreRocks2_1/frame_0006.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0012.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0018.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0024.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0030.jpg",
]

views2 = []

for i, img in enumerate(batch2):
    if i == len(batch2)-1:
        views2.append({
            "img": np.array(Image.open(img).convert("RGB")),
            "intrinsics": intrinsics,
            "is_metric_scale": torch.tensor([True], device=device),
        })
    else:
        views2.append({
            "img": np.array(Image.open(img).convert("RGB")),
            "intrinsics": intrinsics,
            "camera_poses": poses[-4:][i], 
            "is_metric_scale": torch.tensor([True], device=device),
        })


views2_1 = preprocess_inputs(views2)
# Run inference
preds2 = model.infer(
    views2_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

# --- SET YOUR FILTER DISTANCE HERE ---
# This will remove points farther than this distance (in meters/units)
# from the camera's origin.
# MAX_FILTER_DISTANCE = 15.0 

pred = preds2[len(preds2)-1]
    
# Get raw points, colors, and the pose
points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

# --- Filtering---
camera_origin = camera_pose[:3, 3]
distances = np.linalg.norm(points_world - camera_origin, axis=1)
mask = distances <= MAX_FILTER_DISTANCE
filtered_points = points_world[mask]
filtered_colors = colors[mask]


pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_points)
pcd.colors = o3d.utility.Vector3dVector(filtered_colors)


camera_center = camera_pose[:3, 3]
camera_positions.append(camera_center)


camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
camera_frame.transform(camera_pose)
camera_frame.transform(transform_matrix)
pcd.transform(transform_matrix)
geometries.append(pcd)
geometries.append(camera_frame)

   

# # Draw the camera path
# if len(camera_positions) > 1:
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
#     camera_path.paint_uniform_color([1, 0, 0]) # Red path
    
#     geometries.append(camera_path)

# # Apply the final coordinate transform

IndexError: list index out of range